In [1]:
base_dir = './'

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install torch torchvision medmnist matplotlib seaborn scikit-learn tqdm trl datasets transformers opencv-python -q

In [ ]:
# Imports
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import functools
from torch.utils.data import DataLoader
from torchvision import transforms, models
import medmnist
import cv2
from medmnist import INFO
import matplotlib.pyplot as plt


# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"MedMNIST version: {medmnist.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device selected: {device}")

In [ ]:
n_classes = len(INFO['dermamnist']['label'])
class_names = {i: INFO['dermamnist']['label'][str(i)] for i in range(n_classes)}
DataClass = getattr(medmnist, INFO['dermamnist']['python_class'])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # mean from ImageNet dataset
                         std=[0.229, 0.224, 0.225]),    # std from ImageNet dataset
])

test_dataset = DataClass(split='test', transform=transform, download=True, size=224)
test_loader = DataLoader(test_dataset, shuffle=False)

In [ ]:
path = glob.glob(os.path.join(base_dir, "vit-*.pth"))[0]

vit_model = models.vit_b_16()
in_features = vit_model.heads.head.in_features
vit_model.heads.head = nn.Sequential(
    nn.Identity(),
    nn.Linear(in_features, n_classes)
)
vit_model = vit_model.to(device)

vit_model.load_state_dict(torch.load(path, map_location='cuda'))
vit_model.eval()

In [ ]:
resnet_path = glob.glob(os.path.join(base_dir, "resnet-*.pth"))[0]

resnet_model = models.resnet18()
in_features = resnet_model.fc.in_features
resnet_model.fc = nn.Sequential(
    nn.Identity(),
    nn.Linear(in_features, n_classes)
)
resnet_model = resnet_model.to(device)

resnet_model.load_state_dict(torch.load(resnet_path, map_location='cuda'))
resnet_model.eval()

## Part 2: Model interpretability (10 points)

In the previous part, you fine-tuned ResNet-18 and ViT-Base to classify skin conditions using the DermaMNIST dataset.

In healthcare, a "black-box" model that makes correct prediction for the wrong reasons is dangerous. For instance, a model might learn to rely on artifacts specific to the lighting of an hospital room rather than the skin lesion itself. To trust these models, clinicians and engineers must understand what the model is looking at and where it is focusing its attention.

The purpose of this part is to delve more into model interpretability. You will implement and apply the following techniques:
- GradCAM: to visualize convolutional features in ResNet-18.
- Attention rollout: to inspect the self-attention mechanisms of the ViT.

### Your Tasks:

**T1. Qualitative Case Study**

Select 5 diverse images from the test set of DermaMNIST that meet the following criteria:
- Image A: an image where both models predict the correct class with high confidence.
- Image B: an image where both models predict the correct class, but with low confidence (e.g., probability between 0.4 and 0.6)
- Image C: an image where the CNN is correct but ViT is incorrect.
- Image D: an image where ViT is correct but the CNN is incorrect.
- Image E: any image containing an artifact (e.g., a hair, ink markings).

**T2. Implementation of interpretability techniques**

Implement and apply the following visualization methods to your fine-tuned models:
* Grad-CAM: To visualize the gradients and feature maps of the final convolutional layers in ResNet-18.
* Attention Rollout: To aggregate and inspect the self-attention weights across the multiple heads and layers of the Vision Transformer.

**T3. Comparative Analysis**
Generate activation/attention maps for these five cases using your models from Part 1. Compare the heatmaps and analyze how the architectural differences (local receptive fields vs. global self-attention) manifest in the visualizations.

---
**Grading**
You can gain up to 10 points (5 for the implementation and 5 for the analyses) if your code is correct and you give a clear and detailed explanation of your findings. Some questions that might guide your analysis are as following. What did you observe regarding the differences in how ResNet-18 and ViT-Base perceive the images? Did the ViT-Base capture more global context than the CNN? In Image B, is the focus on the lesion even with low confidence? For Images C and D analyze the failures: was the CNN missing global context, or did the ViT get distracted by background/artifacts? For Image E, did the model focus on the artifact instead of the lesion?

#### Section 2.1 - Image selection

In [ ]:
def get_out(model, loader, device):
    model.eval()
    images, probs, labels = [], [], []
    with torch.no_grad():
        for img, label in loader:
            images.append(img)
            labels.append(label)
            probs.append(torch.softmax(model(img.to(device)), dim=1).cpu())
    return torch.cat(images), torch.cat(probs), torch.cat(labels).view(-1)

r_img, r_prob, labels = get_out(resnet_model, test_loader, device)
_, v_prob, _ = get_out(vit_model, test_loader, device)

In [ ]:
r_pred, r_conf = r_prob.argmax(1), r_prob.max(1).values
v_pred, v_conf = v_prob.argmax(1), v_prob.max(1).values

idx = {
    'A': torch.where((r_pred == labels) & (v_pred == labels) & (r_conf > 0.8) & (v_conf > 0.8))[0][0].item(),
    'B': torch.where((r_pred == labels) & (v_pred == labels) & (r_conf < 0.6) & (v_conf < 0.6))[0][0].item(),
    'C': torch.where((r_pred == labels) & (v_pred != labels))[0][0].item(),
    'D': torch.where((v_pred == labels) & (r_pred != labels))[0][0].item(),
    'E': 376
}

idx

In [ ]:
selected_examples = {}

for k, i in idx.items():
    if i is not None:
        selected_examples[k] = {
            'image': r_img[i],
            'label': int(labels[i]),
            'r_p': int(r_pred[i]),
            'v_p': int(v_pred[i]),
            'r_c': round(float(r_conf[i]), 4),
            'v_c': round(float(v_conf[i]), 4)
        }

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (key, ex) in zip(axes, selected_examples.items()):
    img = ex['image'].numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.clip(std * img + mean, 0, 1)

    ax.imshow(img)
    ax.set_title(
        f"{key} | y={class_names[ex['label']]}\n"
        f"R:{class_names[ex['r_p']]} ({ex['r_c']:.2f})\n"
        f"V:{class_names[ex['v_p']]} ({ex['v_c']:.2f})",
        fontsize=8
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

#### Section 2.2 - GradCAM implementation

GradCAM uses the gradients of the target class flowing into the final convolutional layer to produce a coarse localization map highlighting important regions in the image.

**Your task:** Implement a function or class that generate GradCAM heatmaps for your selected images using the fine-tuned ResNet-18 model. Overlay these heatmaps on the original images and look at where the ResNet-18 model is focusing on. Some interesting questions you might ask yourself are whether the heatmaps align with the actual lesions and whether the background or some artifacts affect the predictions.

**Hint:** Review Lecture 7.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.target_layer.register_forward_hook(self._save_act)
        self.target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o): self.activations = o.detach()
    def _save_grad(self, m, gi, go): self.gradients = go[0].detach()

    def generate(self, x):
        logits = self.model(x.unsqueeze(0))
        pred_class = logits.argmax(dim=1).item()
        
        self.model.zero_grad()
        logits[0, pred_class].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1)).squeeze()
        
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        
        return cam.cpu().numpy(), pred_class

def denormalize(t):
    img = t.permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)

gcam = GradCAM(resnet_model, resnet_model.layer4[-1])
fig, axes = plt.subplots(5, 2, figsize=(10, 20))

for i, key in enumerate(['A', 'B', 'C', 'D', 'E']):
    img_t = selected_examples[key]['image']
    true_label = class_names[selected_examples[key]['label']]
    
    heatmap, pred_idx = gcam.generate(img_t.to(device))
    pred_label = class_names[pred_idx]
    
    original = denormalize(img_t)
    heatmap_res = cv2.resize(heatmap, (original.shape[1], original.shape[0]))
    
    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f"Original: {true_label}")
    axes[i, 1].imshow(original)
    axes[i, 1].imshow(heatmap_res, cmap='jet', alpha=0.4)
    axes[i, 1].set_title(f"GradCAM: {pred_label}")
    [ax.axis('off') for ax in axes[i]]

plt.tight_layout()
plt.show()

#### Section 2.3 - ViT attention rollout

Unlike CNNs, ViTs split an image into patches and use self-attention to relate them. Looking at the attention weights of just the last layer is often insufficient. We will use Attention Rollout, which recursively multiplies the attention matrices across all layers to approximate the contribution of each patch to the final decision.

**Your tasks:**
- Use the provided `ViTAttentionRollout` class to capture attention weights from all layers of the ViT model
- Implement the `compute_rollout(attention_maps)` function to aggregate these weights (a combination of matrix multiplication and adding residual connections)
- Visualize the resulting rollout maps overlaid to the 5 original images you selected and analyse what the ViT is focusing on.

**Hint**:
- Review Lecture 8.
- Follow these steps for the implementation of the `compute_rollout(attention_maps)` function:
    1. Average the attention maps across the head dimension.
    2. Create the identity matrix with shape $\text{num\_tokens} \times \text{num\_tokens}$
    3. For each attention matrix at Step 1:
          * Add the residual connection ($0.5 \cdot I + 0.5 \cdot \text{attention matrix}$)
          * Multiply with the accumulated rollout from the previous layers.

In [ ]:
# This cell contains some helper functions to capture attention weights from the ViT layers and visualize the rollout.
# This is slightly different than what presented in Lecture 8, but it integrates well with models downloaded from Hugging Face.
# Do not modify it, unless necessary.

class ViTAttentionRollout:
    def __init__(self, model):
        self.model = model
        self.model.eval() # Ensure model is in eval mode
        self.attention_maps = []
        self.hooks = []

        for i, layer in enumerate(model.encoder.layers):
            hook = layer.self_attention.register_forward_hook(self.get_attention_hook(i))
            self.hooks.append(hook)

    def get_attention_hook(self, layer_id):
        def hook(module, input, output):
            self.attention_maps.append(output[1].detach().cpu()) # Capture weights
        return hook

    def __call__(self, img_tensor): # img_tensor must be with shape [batch_size, channels, width, hight]
        self.attention_maps = []
        with torch.no_grad():
            _ = self.model(img_tensor)
        return self.attention_maps

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()


# Necessary to force weight calculation (which is disabled in torchvision models to speed up the calculation)
def forward_with_attn_weights(self, query, key, value, key_padding_mask=None, need_weights=False, attn_mask=None, average_attn_weights=True):
    return torch.nn.MultiheadAttention.forward(
        self, query, key, value, key_padding_mask=key_padding_mask,
        need_weights=True,  # Force weights calculation
        attn_mask=attn_mask,
        average_attn_weights=average_attn_weights
    )

# Apply the patch to every encoder layer
for block in vit_model.encoder.layers:
    block.self_attention.forward = functools.partial(forward_with_attn_weights, block.self_attention)

#---- Visualization functions ----
def denormalize(tensor):
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    return np.clip(inv_normalize(tensor).cpu().permute(1, 2, 0).numpy(), 0, 1)

def visualize_vit_rollout(img, y, pred_class, vit_rollout):
    img_np = denormalize(img)

    # Resize ViT rollout to 224
    # Rollout is flat (num_patches), need to reshape to grid
    grid_size = int(np.sqrt(vit_rollout.shape[0]))
    vit_map = vit_rollout.reshape(grid_size, grid_size).numpy()
    vit_map_resized = torch.nn.functional.interpolate(
        torch.from_numpy(vit_map).unsqueeze(0).unsqueeze(0).float(),
        size=(224, 224), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    fig, axs = plt.subplots(1, 2, figsize=(6, 3))

    axs[0].imshow(img_np)
    axs[0].axis('off')
    axs[0].set_title(f"Original Image\n(True class: {y[0]})", fontsize=12)

    axs[1].imshow(img_np)
    axs[1].imshow(vit_map_resized, cmap='jet', alpha=0.4)
    axs[1].axis('off')
    axs[1].set_title(f"ViT attention rollout\n(Pred class: {pred_class})", fontsize=12)

    plt.tight_layout()
    plt.savefig(os.path.join(base_dir, f"rollout_visualization.png"), dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
def compute_rollout(attention_maps):
    num_tokens = attention_maps[0].shape[-1]
    rollout = torch.eye(num_tokens)

    for attn in attention_maps:
        a = attn[0].mean(dim=0)
        a_hat = 0.5 * (a + torch.eye(num_tokens))
        a_hat /= a_hat.sum(dim=-1, keepdim=True)
        rollout = a_hat @ rollout

    return rollout[0, 1:]


rollout_runner = ViTAttentionRollout(vit_model)

for i, key in enumerate(['A', 'B', 'C', 'D', 'E']):
    ex = selected_examples[key]
    img_t = ex['image'].unsqueeze(0).to(device)
    
    attn_maps = rollout_runner(img_t)
    rollout_vec = compute_rollout(attn_maps)
    
    visualize_vit_rollout(
        ex['image'],
        [class_names[ex['label']]],
        class_names[ex['v_p']],
        rollout_vec
    )

rollout_runner.remove_hooks()